In [ ]:
import sys, os
sys.path.insert(0, '../utils')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from utils import load_neurons_table, load_synapses_position_transformed
from connectome_types import CONNECTOME_SYN_TABLE_PATH, CONNECTOME_PRE_SYN_TABLE_PATH
from neuron_custom_features import calc_spines_features
from spines_utils import filter_valid_neuron_w_spines, split_syn_mat_by_type_four
from plot_utils import ex_color, inh_color
from connectome_types import SPINE_TABLE_OUTGOING
import matplotlib.lines as mlines
from matplotlib.gridspec import GridSpec
from stats_corr import p_to_stars, add_log_curve, add_reg_line
from spine_pref_utils import per_neuron_spine_ratio, plot_spine_along_axon, build_subnet_spine_ratio_df
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

from matplotlib.ticker import PercentFormatter
from mpl_toolkits.mplot3d import Axes3D
from figures_utils import add_panel_label
from matplotlib.ticker import FormatStrFormatter

import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.ticker import MultipleLocator
from figures_utils import plot_nested_cylinders

In [ ]:
neurons_df = load_neurons_table()
syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_SYN_TABLE_PATH)
df, syn_with_tags = calc_spines_features(neurons_df, syn_df)
neuron_clf_type = df[['root_id', 'clf_type']].set_index('root_id').to_dict(orient='index')
df, filtered_syn_mat, filtered_bin_mat, filtered_mapping, filtered_reverse_mapping, ex_neurons, inh_neurons = filter_valid_neuron_w_spines(df)

spine_df_outgoing = pd.read_csv(SPINE_TABLE_OUTGOING)
outgoing_syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_PRE_SYN_TABLE_PATH)
outgoing_syn_with_tags = outgoing_syn_df[outgoing_syn_df.id_.isin(spine_df_outgoing.target_id)].copy()
outgoing_syn_with_tags['tag'] = outgoing_syn_with_tags.id_.map(spine_df_outgoing.set_index('target_id').tag)
print(f'neurons post: {outgoing_syn_with_tags.post_id.nunique()}')
print(f'neurons pre: {outgoing_syn_with_tags.pre_id.nunique()}')
print(f'outgoing synapses with tags: {outgoing_syn_with_tags.shape[0]}')

ex_outgoing_syn_with_tags = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_clf_type == 'E']
inh_outgoing_syn_with_tags = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_clf_type == 'I']
print(f'ex outgoing synapses with tags: {ex_outgoing_syn_with_tags.shape[0]}')
print(f'inh outgoing synapses with tags: {inh_outgoing_syn_with_tags.shape[0]}')

EE, EI, IE, II, ex_idx, inh_idx = split_syn_mat_by_type_four(filtered_bin_mat, filtered_mapping, neuron_clf_type)
print(f"Block shapes — EE:{EE.shape}  EI:{EI.shape}  IE:{IE.shape}  II:{II.shape}")

ex_syn_tags = syn_with_tags[syn_with_tags.pre_clf_type == 'E']
ee_syn_tags = ex_syn_tags[ex_syn_tags.post_clf_type == 'E']
ei_syn_tags = ex_syn_tags[ex_syn_tags.post_clf_type == 'I']

inh_syn_tags = syn_with_tags[syn_with_tags.pre_clf_type == 'I']
ii_syn_tags = inh_syn_tags[inh_syn_tags.post_clf_type == 'I']
ie_syn_tags = inh_syn_tags[inh_syn_tags.post_clf_type == 'E']

ex_neurons  = ex_neurons.copy()
inh_neurons = inh_neurons.copy()

In [ ]:
main_feature = 'spine'
USE_BINARY_DEGREE = False

root_ids = ex_neurons.root_id.tolist()
ee_stats  = per_neuron_spine_ratio(ee_syn_tags,  root_ids)
e_all_stats = per_neuron_spine_ratio(ex_syn_tags,  root_ids)
e_all_outside_stats = per_neuron_spine_ratio(spine_df_outgoing, root_ids, group_by='pre_pt_root_id')

ii_stats  = per_neuron_spine_ratio(ii_syn_tags,  inh_neurons.root_id.tolist())
i_all_stats = per_neuron_spine_ratio(inh_syn_tags,  inh_neurons.root_id.tolist())   
i_all_outside_stats = per_neuron_spine_ratio(spine_df_outgoing, inh_neurons.root_id.tolist(), group_by='pre_pt_root_id')


if USE_BINARY_DEGREE:
    ee_outdegree    = {filtered_mapping[ex_idx[n]]: int(EE[n, :].sum())
                        for n in range(EE.shape[0])}
    e_all_outdegree = {filtered_mapping[ex_idx[n]]: int(EE[n, :].sum() + EI[n, :].sum())
                        for n in range(EE.shape[0])}

    ii_outdegree    = {filtered_mapping[inh_idx[n]]: int(II[n, :].sum())
                        for n in range(II.shape[0])}
    i_all_outdegree = {filtered_mapping[inh_idx[n]]: int(II[n, :].sum() + IE[n, :].sum())
                        for n in range(II.shape[0])}

    ex_neurons['x_EE']  = ex_neurons.root_id.map(ee_outdegree).fillna(0).astype(int)
    ex_neurons['x_all'] = ex_neurons.root_id.map(e_all_outdegree).fillna(0).astype(int)

    inh_neurons['x_II']  = inh_neurons.root_id.map(ii_outdegree).fillna(0).astype(int)
    inh_neurons['x_all'] = inh_neurons.root_id.map(i_all_outdegree).fillna(0).astype(int)
else:
    ex_neurons['x_EE']  = ex_neurons.root_id.map(ee_stats['n_syn'])
    ex_neurons['x_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['n_syn'])

    inh_neurons['x_II']  = inh_neurons.root_id.map(ii_stats['n_syn'])
    inh_neurons['x_all_outside'] = inh_neurons.root_id.map(i_all_outside_stats['n_syn'])


ex_neurons[f'outgoing_{main_feature}_ratio_EE']  = ex_neurons.root_id.map(ee_stats['ratio'])
ex_neurons[f'outgoing_{main_feature}_ratio_all'] = ex_neurons.root_id.map(e_all_stats['ratio'])
ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['ratio'])

inh_neurons[f'outgoing_{main_feature}_ratio_II']  = inh_neurons.root_id.map(ii_stats['ratio'])
inh_neurons[f'outgoing_{main_feature}_ratio_all'] = inh_neurons.root_id.map(i_all_stats['ratio'])
inh_neurons[f'outgoing_{main_feature}_ratio_all_outside'] = inh_neurons.root_id.map(i_all_outside_stats['ratio'])

In [ ]:
df_EE = build_subnet_spine_ratio_df(EE, ex_idx,  ex_idx,  filtered_mapping, 'E','E', syn_with_tags, ex_neurons)
df_EI = build_subnet_spine_ratio_df(EI, ex_idx,  inh_idx, filtered_mapping, 'E','I', syn_with_tags, ex_neurons)
df_IE = build_subnet_spine_ratio_df(IE, inh_idx, ex_idx,  filtered_mapping, 'I','E', syn_with_tags, inh_neurons)
df_II = build_subnet_spine_ratio_df(II, inh_idx, inh_idx, filtered_mapping, 'I','I', syn_with_tags, inh_neurons)

# Plot

In [ ]:
plt.rcParams['font.size'] = 13
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['font.family'] = 'Arial'

scatter_overlay_fontsize = 11
scatter_overlay_fontsize_small = 9

spiny_color =  '#7C3AED' 
aspiny_color = '#059669'


axon_color = '#2d2d2d'      
circle_size = 5      
alpha_synapse = 0.6     
axon_lw = 4           
axon_alpha = 0.8      
dend_lw = 1.2           
dend_alpha = 0.8        

In [ ]:
from plot_utils import plot_skeleton_continuous
from sk_load_utils import load_clean_sk, load_col_skeleton_only


def plot_single_neuron(ax, sk_a, sk_d, syn_spines, syn_nonspines, dend_color,
                       z_spine=5, z_shaft=4, z_order_dend=1, circle_size=6, alpha_synapse=0.6,
                       axon_lw=4, dend_lw=4, axon_alpha=0.8, dend_alpha=0.8, axon_color='b'):
    plot_skeleton_continuous(ax=ax, sk=sk_a, lw=axon_lw, alpha=axon_alpha, 
                             color=axon_color, ignore_vertex_zero=True, coords=('x', 'y'))
    
    ax.scatter(syn_nonspines.pt_position_xt, syn_nonspines.pt_position_yt, 
               s=circle_size, alpha=alpha_synapse, marker='o', color=aspiny_color, zorder=z_shaft)

    ax.scatter(syn_spines.pt_position_xt, syn_spines.pt_position_yt, 
               s=circle_size, alpha=alpha_synapse, marker='o', color=spiny_color, zorder=z_spine)
    
    plot_skeleton_continuous(ax=ax, sk=sk_d, lw=dend_lw, alpha=dend_alpha, 
                             color=dend_color, ignore_vertex_zero=True, coords=('x', 'y'), zorder=z_order_dend)
    
    ax.invert_yaxis()

In [ ]:
high_degree_neuron = ex_neurons[ex_neurons.root_id == 864691135617152361]
low_degree_neuron = ex_neurons[ex_neurons.root_id == 864691136177632518]
inh_neuron = inh_neurons[inh_neurons.root_id == 864691135562001633]

high_deg_nodes = [864691135617152361, 864691136177632518]
inh_nodes = [864691135562001633]

In [ ]:
node_id = 864691135617152361
sk_dent = load_col_skeleton_only(node_id, reset_axon=True, old=True)
sk_axon = load_col_skeleton_only(node_id, reset_dentrites=True, old=True)

subset = 'outside'
if subset == 'outside':
    outgoing_syn = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_id == node_id]
elif subset == 'e-to-all':
    outgoing_syn = ex_syn_tags[ex_syn_tags.pre_id == node_id]
elif subset == 'ee':
     outgoing_syn = ee_syn_tags[ee_syn_tags.pre_id == node_id]
print(f"Total outgoing synapses for root_id {node_id}: {len(outgoing_syn)}")

outgoing_syn_onto_spines = outgoing_syn[outgoing_syn.tag == 'spine']
print(f"Total outgoing synapses onto spines for root_id {node_id}: {len(outgoing_syn_onto_spines)}")
outgoing_syn_onto_nonspines = outgoing_syn[outgoing_syn.tag != 'spine']
print(f"Total outgoing synapses onto non-spines for root_id {node_id}: {len(outgoing_syn_onto_nonspines)}")

spine_pref = len(outgoing_syn_onto_spines) / len(outgoing_syn) if len(outgoing_syn) > 0 else 0
print(f"Spine preference for root_id {node_id}: {spine_pref:.2f}")

In [ ]:
low_node_id =  864691136177632518 # 6pct
low_sk_dent = load_col_skeleton_only(low_node_id, reset_axon=True, old=True)
low_sk_axon = load_col_skeleton_only(low_node_id, reset_dentrites=True, old=True)

subset = 'outside'
if subset == 'outside':
    low_outgoing_syn = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_id == low_node_id]
elif subset == 'e-to-all':
    low_outgoing_syn = ex_syn_tags[ex_syn_tags.pre_id == low_node_id]
elif subset == 'ee':
     low_outgoing_syn = ee_syn_tags[ee_syn_tags.pre_id == low_node_id]
print(f"Total outgoing synapses for root_id {low_node_id}: {len(low_outgoing_syn)}")

low_outgoing_syn_onto_spines = low_outgoing_syn[low_outgoing_syn.tag == 'spine']
print(f"Total outgoing synapses onto spines for root_id {low_node_id}: {len(low_outgoing_syn_onto_spines)}")
low_outgoing_syn_onto_nonspines = low_outgoing_syn[low_outgoing_syn.tag != 'spine']
print(f"Total outgoing synapses onto non-spines for root_id {low_node_id}: {len(low_outgoing_syn_onto_nonspines)}")

low_spine_pref = len(low_outgoing_syn_onto_spines) / len(low_outgoing_syn) if len(low_outgoing_syn) > 0 else 0
print(f"Spine preference for root_id {low_node_id}: {low_spine_pref:.2f}")

In [ ]:
inh_node_id =  864691135562001633 
inh_sk_dent = load_col_skeleton_only(inh_node_id, reset_axon=True, old=True)
inh_sk_axon = load_col_skeleton_only(inh_node_id, reset_dentrites=True, old=True)

subset = 'outside'
if subset == 'outside':
    inh_outgoing_syn = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_id == inh_node_id]
    inh_outgoing_syn = inh_outgoing_syn.sample(frac=0.1, random_state=42)
print(f"Total outgoing synapses for root_id {inh_node_id}: {len(inh_outgoing_syn)}")

inh_outgoing_syn_onto_spines = inh_outgoing_syn[inh_outgoing_syn.tag == 'spine']
print(f"Total outgoing synapses onto spines for root_id {inh_node_id}: {len(inh_outgoing_syn_onto_spines)}")
inh_outgoing_syn_onto_nonspines = inh_outgoing_syn[inh_outgoing_syn.tag != 'spine']
print(f"Total outgoing synapses onto non-spines for root_id {inh_node_id}: {len(inh_outgoing_syn_onto_nonspines)}")

inh_spine_pref = len(inh_outgoing_syn_onto_spines) / len(inh_outgoing_syn) if len(inh_outgoing_syn) > 0 else 0
print(f"Spine preference for root_id {inh_node_id}: {inh_spine_pref:.2f}")

In [ ]:
def add_custom_legend(ax, colors, labels, markersize=6, **kwargs):
    handles = [
        Line2D([0], [0], marker='o', color=c, linestyle='None', label=l, markerfacecolor=c, markersize=markersize)
        for c, l in zip(colors, labels)
    ]
    ax.legend(handles=handles, **kwargs, frameon=False)

def plot_legend_EI_clf_type(ax, labels=['E', 'I'], colors=[ex_color, inh_color], 
                             fontsize='small', markersize=6): 
    legend_elements = [
        Line2D([0], [0], marker='o', color=colors[i], markerfacecolor=colors[i], 
               linestyle='None', markersize=markersize, label=labels[i])
        for i in range(len(labels))
    ]

    # UPDATE HERE: Add bbox_to_anchor to push the legend upwards
    ax.legend(handles=legend_elements, frameon=False,
            title='',
            loc='lower center',         # Anchor the bottom center of the legend...
            bbox_to_anchor=(0.5, 1.2),  # ...to a point slightly above the axis (y=1.2)
            ncol=2,
            fontsize=fontsize) 

    ax.set_axis_off()

In [ ]:
fig = plt.figure(figsize=(20, 22), dpi=600)
gs_master = fig.add_gridspec(5, 1, height_ratios=[2.5, 1, 1, 1, 1], hspace=0.4)

# ────────────────────────────────────────────────────────────────────────
# --- ROW 0: Morphology (Top Row) ---
gs_morph = gs_master[0].subgridspec(1, 3, wspace=0.0)
axes_morph = [fig.add_subplot(gs_morph[i]) for i in range(3)]

plot_single_neuron(ax=axes_morph[0], sk_a=sk_axon, sk_d=sk_dent,
                   syn_spines=outgoing_syn_onto_spines, syn_nonspines=outgoing_syn_onto_nonspines,
                   circle_size=12, axon_color=ex_color, dend_color='black')

plot_single_neuron(ax=axes_morph[1], sk_a=low_sk_axon, sk_d=low_sk_dent,
                   syn_spines=low_outgoing_syn_onto_spines, syn_nonspines=low_outgoing_syn_onto_nonspines,
                   circle_size=12, axon_color=ex_color, dend_color='black')

plot_single_neuron(ax=axes_morph[2],
                   sk_a=inh_sk_axon, sk_d=inh_sk_dent,
                   syn_spines=inh_outgoing_syn_onto_spines, syn_nonspines=inh_outgoing_syn_onto_nonspines,
                   axon_lw=3, dend_lw=2, dend_alpha=0.9, axon_alpha=0.5,
                   circle_size=12, alpha_synapse=0.6, z_shaft=6, z_order_dend=5,
                   axon_color=inh_color, dend_color='black')

legend_elements = [
    Line2D([0], [0], marker='o', color=spiny_color, label='onto spine',
           markerfacecolor=spiny_color, markersize=12, linestyle='None'),
    Line2D([0], [0], marker='o', color=aspiny_color, label='onto shaft/soma',
           markerfacecolor=aspiny_color, markersize=12, linestyle='None')
]
axes_morph[2].legend(handles=legend_elements, loc='upper right', frameon=False, fontsize=14, title='Whole volume synapses')

# --- Match physical scale for the two LEFT neurons only ---
left_axes = [axes_morph[0], axes_morph[1]]
left_lims = []
for ax in left_axes:
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    left_lims.append({
        'x_center': sum(xlim) / 2,
        'y_center': sum(ylim) / 2,
        'x_span': abs(xlim[1] - xlim[0]),
        'y_span': abs(ylim[1] - ylim[0]),
        'y_inverted': ylim[0] > ylim[1]
    })

max_left_x_span = max(lim['x_span'] for lim in left_lims)
max_left_y_span = max(lim['y_span'] for lim in left_lims)

for ax, lim in zip(left_axes, left_lims):
    ax.set_xlim(lim['x_center'] - max_left_x_span / 2, lim['x_center'] + max_left_x_span / 2)
    if lim['y_inverted']:
        ax.set_ylim(lim['y_center'] + max_left_y_span / 2, lim['y_center'] - max_left_y_span / 2)
    else:
        ax.set_ylim(lim['y_center'] - max_left_y_span / 2, lim['y_center'] + max_left_y_span / 2)

# --- Per-axis bottom cropping: [axes_morph[0], axes_morph[1], axes_morph[2]] ---
crop_bottom_fractions = [0.15, 0.15, 0.30]
if len(crop_bottom_fractions) != len(axes_morph):
    raise ValueError('crop_bottom_fractions must match number of axes.')

for ax, crop_bottom_fraction in zip(axes_morph, crop_bottom_fractions):
    y0, y1 = ax.get_ylim()
    y_span = abs(y1 - y0)
    crop_amount = y_span * crop_bottom_fraction
    if y0 > y1:  # inverted axis: y0 is the visual bottom
        ax.set_ylim(y0 - crop_amount, y1)
    else:
        ax.set_ylim(y0 + crop_amount, y1)

# --- Scale bars: ---
fontprops_main = fm.FontProperties(size=14)
scalebar_left = AnchoredSizeBar(transform=axes_morph[0].transData, size=100, label='100 µm', sep=5,
                                loc='lower right', pad=0.5, color='black', frameon=False,
                                size_vertical=2, fontproperties=fontprops_main,
                                bbox_to_anchor=(0.45, 0.02), bbox_transform=axes_morph[0].transAxes)
scalebar_mid = AnchoredSizeBar(transform=axes_morph[1].transData, size=100, label='100 µm', sep=5,
                                loc='lower right', pad=0.5, color='black', frameon=False,
                                size_vertical=2, fontproperties=fontprops_main,
                                bbox_to_anchor=(0.65, 0.02), bbox_transform=axes_morph[1].transAxes)
scalebar_right = AnchoredSizeBar(transform=axes_morph[2].transData, size=75, label='75 µm', sep=5,
                                 loc='lower right', pad=0.5, color='black', frameon=False,
                                 size_vertical=1.2, fontproperties=fontprops_main,
                                 bbox_to_anchor=(0.775, 0.03), bbox_transform=axes_morph[2].transAxes)

for ax in axes_morph:
    ax.margins(0)
    ax.set_axis_off()
    ax.patch.set_alpha(0)
    
axes_morph[0].add_artist(scalebar_left)
axes_morph[1].add_artist(scalebar_mid)
axes_morph[2].add_artist(scalebar_right)

# --- Per-axis '=' labels ---
equals_x_positions = [0.31, 0.43, 0.5]
equals_y_positions  = [0.01, 0.01, 0.018]
for ax, eq_x, eq_y in zip(axes_morph, equals_x_positions, equals_y_positions):
    ax.text(eq_x, eq_y, '=', transform=ax.transAxes,
            ha='right', va='top', fontsize=22, color='black', clip_on=False)

# ────────────────────────────────────────────────────────────────────────
# --- ROW 1: 5 Columns ---
cyl_width = 0.75
wspace = 0.5
gs_top = gs_master[1].subgridspec(1, 5, width_ratios=[cyl_width, 1, 1, 1, 1], wspace=wspace)

axes_top = []
gs_cyl = gs_top[0].subgridspec(2, 1, height_ratios=[0.05, 1], hspace=0.0)
ax_legend = fig.add_subplot(gs_cyl[0])
axes_top.append(fig.add_subplot(gs_cyl[1], projection='3d'))
for i in range(1, 5):
    axes_top.append(fig.add_subplot(gs_top[i]))

# Cylinder 1 (show_outer=True)
axes_top[0].set_title("Whole volume")
plot_nested_cylinders(axes_top[0], show_outer=True)
plot_legend_EI_clf_type(ax=ax_legend, markersize=4)

# Scatter E 1 (Index 1)
axes_top[1].set_title("E → All")
axes_top[1].scatter(x=ex_neurons['axon_length'], y=ex_neurons['x_all_outside'], edgecolors=ex_color, 
                    facecolors='none', s=5, alpha=0.7, zorder=5)
add_reg_line(ex_neurons['axon_length'], ex_neurons['x_all_outside'], ax=axes_top[1], color='gray',
             add_reg_text='r_only', reg_text_font_size=scatter_overlay_fontsize, linewidth=1.2)

# Scatter E 2 (Index 2)
axes_top[2].scatter(x=ex_neurons['x_all_outside'], y=ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'],
                    edgecolors=ex_color, facecolors='none', s=5, alpha=0.7)
add_log_curve(ex_neurons, 'x_all_outside', f'outgoing_{main_feature}_ratio_all_outside', axes_top[2],
              color='gray', fontsize=scatter_overlay_fontsize, linestyle=':')

# Scatter I 1 (Index 3)
axes_top[3].set_title("I → All")
axes_top[3].scatter(x=inh_neurons['axon_length'], y=inh_neurons['x_all_outside'], edgecolors=inh_color, 
                    facecolors='none', s=5, alpha=0.7)
add_reg_line(inh_neurons['axon_length'], inh_neurons['x_all_outside'], ax=axes_top[3], color='gray',
             add_reg_text='r_only', reg_text_font_size=scatter_overlay_fontsize, linewidth=1.2)

# Scatter I 2 (Index 4)
axes_top[4].scatter(x=inh_neurons['x_all_outside'], y=inh_neurons[f'outgoing_{main_feature}_ratio_all_outside'],
                    edgecolors=inh_color, facecolors='none', s=5, alpha=0.7)
add_log_curve(inh_neurons, 'x_all_outside', f'outgoing_{main_feature}_ratio_all_outside', axes_top[4],
              color='gray', fontsize=scatter_overlay_fontsize, linestyle=':')

# Sample neurons
for ax, x_feat, y_feat in zip([axes_top[1], axes_top[2]], ['axon_length', 'x_all_outside'], ['x_all_outside', f'outgoing_{main_feature}_ratio_all_outside']):
    ax.scatter(x=high_degree_neuron[x_feat], y=high_degree_neuron[y_feat], color=spiny_color, s=35, zorder=10, label='Neuron in A')
    ax.scatter(x=low_degree_neuron[x_feat], y=low_degree_neuron[y_feat], color=aspiny_color, s=35, zorder=10, label='Neuron in B')

for ax, x_feat, y_feat in zip([axes_top[3], axes_top[4]], ['axon_length', 'x_all_outside'], ['x_all_outside', f'outgoing_{main_feature}_ratio_all_outside']):
    ax.scatter(x=inh_neuron[x_feat], y=inh_neuron[y_feat], color=aspiny_color, s=35, zorder=10, label='Neuron in C')

for i, ax in enumerate(axes_top):
    if i in [1, 2, 3, 4]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    if i in [1, 3]:
        ax.set_xlabel('Total axonal length (µm)')
        ax.set_ylabel('# of outgoing synapses')
        ax.legend(loc='upper left', frameon=False)
        ax.xaxis.set_minor_locator(MultipleLocator(5000))
    if i in [2, 4]:
        ax.set_xlabel('# of outgoing synapses')
        ax.set_ylabel('% of output synapses\n on target spines')
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
    if i == 4:
        ax.xaxis.set_minor_locator(MultipleLocator(2500))


# --- ROW 2: Spanned Columns (Middle Row) ---
gs_mid = gs_master[2].subgridspec(1, 5, width_ratios=[cyl_width, 1, 1, 1, 1], wspace=wspace)
ax_mid_cyl = fig.add_subplot(gs_mid[0, 0], projection='3d')
ax_mid_plt_e = fig.add_subplot(gs_mid[0, 1:3]) 
ax_mid_plt_i = fig.add_subplot(gs_mid[0, 3:5], sharey=ax_mid_plt_e)

ax_mid_cyl.set_title("Whole volume")
plot_nested_cylinders(ax_mid_cyl, show_outer=True)

syn_list    = [ex_outgoing_syn_with_tags, inh_outgoing_syn_with_tags]
pop_colors  = [ex_color, inh_color]
axon_colors = [[spiny_color, aspiny_color], [aspiny_color]]
node_samples= [high_deg_nodes, inh_nodes]
axes_mid_plots = [ax_mid_plt_e, ax_mid_plt_i]

for ax, outgoing_syn, axon_color, pop_color, node_sample in zip(axes_mid_plots, syn_list, axon_colors, pop_colors, node_samples):
    plot_spine_along_axon(
        ax=ax, outgoing_syn=outgoing_syn, node_id_list=node_sample,
        axon_colors=axon_color, pop_color=pop_color,
        show_xlabel=False, show_legend=False,
        cross_dist_textsize=scatter_overlay_fontsize, num_single_bins=16, num_pop_bins=160,
        axon_bin_fontsize=scatter_overlay_fontsize)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
    ax.set_xlabel('Axonal path length from soma (µm)')

add_custom_legend(axes_mid_plots[0], colors=axon_colors[0], labels=['Neuron in A', 'Neuron in B'], loc='lower right')
add_custom_legend(axes_mid_plots[1], colors=axon_colors[1], labels=['Neuron in C'], loc='upper left')
axes_mid_plots[1].axhline(0.5, color='gray', linestyle='--', lw=1.5, alpha=0.7)
axes_mid_plots[0].set_ylabel('% of output synapses\non target spines')
axes_mid_plots[1].set_ylabel('')
ax_mid_plt_e.set_title('E → All')
ax_mid_plt_i.set_title('I → All')


# ────────────────────────────────────────────────────────────────────────
# --- ROW 3: 5 Columns (Fourth Row) ---
gs_bottom = gs_master[3].subgridspec(1, 5, width_ratios=[cyl_width, 1, 1, 1, 1], wspace=wspace)

axes_bottom = []
axes_bottom.append(fig.add_subplot(gs_bottom[0, 0], projection='3d'))
scatter_share_ax = None

for i in range(1, 5):
    if scatter_share_ax is None:
        ax = fig.add_subplot(gs_bottom[0, i])
        scatter_share_ax = ax
    else:
        ax = fig.add_subplot(gs_bottom[0, i], sharey=scatter_share_ax)
    axes_bottom.append(ax)

# Cylinder 3 (show_outer=False)
axes_bottom[0].set_title("Micro column")
plot_nested_cylinders(axes_bottom[0], show_outer=False)

axes_bottom[1].set_ylabel('% of output synapses\non target spines')
axes_bottom[1].yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))


bottom_scatter_configs = [
    (1, df_EE, ex_color, 'E → E'), (2, df_EI, ex_color, 'E → I'),
    (3, df_II, inh_color, 'I → I'), (4, df_IE, inh_color, 'I → E')
]

for idx, df_, color, title in bottom_scatter_configs:
    ax = axes_bottom[idx]
    ax.scatter(x=df_['n_syn'], y=df_['outgoing_spine_ratio'], edgecolors=color, 
               facecolors='none', s=5, alpha=0.7, zorder=5)
    add_log_curve(df_, 'n_syn', 'outgoing_spine_ratio', ax=ax, color='gray', fontsize=scatter_overlay_fontsize, linestyle=':')
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('# of outgoing synapses')

    # Add sample neurons
    if idx in [1, 2]:  # E plots
        high_deg_neuron = df_[df_.root_id == high_deg_nodes[0]]
        low_degree_neuron = df_[df_.root_id == high_deg_nodes[1]]
        ax.scatter(x=high_deg_neuron['n_syn'], y=high_deg_neuron['outgoing_spine_ratio'],
                   color=spiny_color, s=35, zorder=10, label='Neuron in A')
        ax.scatter(x=low_degree_neuron['n_syn'], y=low_degree_neuron['outgoing_spine_ratio'],
                   color=aspiny_color, s=35, zorder=10, label='Neuron in B')
    elif idx in [3, 4]:  # I plots
        inh_neuron = df_[df_.root_id == inh_nodes[0]]
        ax.scatter(x=inh_neuron['n_syn'], y=inh_neuron['outgoing_spine_ratio'],
                   color=aspiny_color, s=35, zorder=10, label='Neuron in C')

axes_bottom[2].legend(loc='upper right', frameon=False)
axes_bottom[3].legend(loc='upper left', frameon=False)

# ────────────────────────────────────────────────────────────────────────
# --- ROW 4: 5 Columns (New Bottom Row) ---
gs_row4 = gs_master[4].subgridspec(1, 5, width_ratios=[cyl_width, 1, 1, 1, 1], wspace=wspace)
block_titles = ['E → E', 'E → I', 'I → I', 'I → E']

axes_row4 = []
axes_row4.append(fig.add_subplot(gs_row4[0, 0], projection='3d'))
row4_share_ax = None

for i in range(1, 5):
    if row4_share_ax is None:
        ax = fig.add_subplot(gs_row4[0, i])
        row4_share_ax = ax
    else:
        ax = fig.add_subplot(gs_row4[0, i], sharey=row4_share_ax)
    axes_row4.append(ax)

axes_row4[0].set_title("Micro column")
plot_nested_cylinders(axes_row4[0], show_outer=False)

# Applying the configuration requested
row4_syn_list = [ee_syn_tags, ei_syn_tags, ii_syn_tags, ie_syn_tags]
row4_pop_colors = [ex_color, ex_color, inh_color, inh_color]
row4_axon_colors = [axon_colors[0], axon_colors[0], axon_colors[1], axon_colors[1]] # Reusing from Row 2
row4_node_samples = [high_deg_nodes, high_deg_nodes, inh_nodes, inh_nodes] # Reusing from Row 2

for i, ax in enumerate(axes_row4[1:]):
    plot_spine_along_axon(
        ax=ax, outgoing_syn=row4_syn_list[i], node_id_list=row4_node_samples[i],
        axon_colors=row4_axon_colors[i], pop_color=row4_pop_colors[i],
        show_xlabel=False, show_legend=False, show_cross_distance=i==0, max_distance=600,
        cross_dist_textsize=scatter_overlay_fontsize_small, num_single_bins=6,
        num_pop_bins=60,
        axon_bin_fontsize=scatter_overlay_fontsize_small)
    
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
    ax.set_xlabel('Axonal path length\nfrom soma (µm)')
    ax.set_ylabel('')
    ax.set_xlim(0, 610)
    ax.set_title(block_titles[i])
    if i >= 1:
        ax.axhline(0.5, color='gray', linestyle='--', lw=1.5, alpha=0.7)

    axes_row4[1].set_ylabel('% of output synapses\non target spines')

# Add custom legends identically to row 2
add_custom_legend(axes_row4[2], colors=axon_colors[0], labels=['Neuron in A', 'Neuron in B'], loc='upper right')
add_custom_legend(axes_row4[3], colors=axon_colors[1], labels=['Neuron in C'], loc='upper left')


# ────────────────────────────────────────────────────────────────────────
# --- Final Cleanup & Adjustments ---

fig.canvas.draw()

delta1 = 0.08  
delta2 = 0.125 

pos0 = axes_morph[0].get_position()
pos1 = axes_morph[1].get_position()
pos2 = axes_morph[2].get_position()

new_x0 = pos0.x0
new_x1 = pos1.x0 - delta1
new_x2 = pos2.x0 - delta1 - delta2

original_total_width = pos2.x1 - pos0.x0
new_total_width = (new_x2 + pos2.width) - pos0.x0
scale = original_total_width / new_total_width

for ax, nx, orig_pos in zip(axes_morph, [new_x0, new_x1, new_x2], [pos0, pos1, pos2]):
    scaled_width = orig_pos.width * scale
    scaled_height = orig_pos.height * scale
    rel_x = nx - pos0.x0
    final_x = pos0.x0 + (rel_x * scale)
    y_center = orig_pos.y0 + (orig_pos.height / 2.0)
    final_y = y_center - (scaled_height / 2.0)
    ax.set_position([final_x, final_y, scaled_width, scaled_height])

axes_morph[0].set_zorder(3)
axes_morph[1].set_zorder(1)
axes_morph[2].set_zorder(2)

add_panel_label(axes_morph[0], 'A')
add_panel_label(axes_morph[1], 'B', xy=(0.275, 1.01))
add_panel_label(axes_morph[2], 'C', xy=(0.325, 0.997))
add_panel_label(axes_top[0], 'D', xy=(-0.01, 1.15))
add_panel_label(axes_top[3], 'E', xy=(-0.05, 1.05))
add_panel_label(ax_mid_cyl, 'F', xy=(-0.01, 1.105))
add_panel_label(ax_mid_plt_i, 'G', xy=(-0.025, 1.05))
add_panel_label(axes_bottom[0], 'H', xy=(-0.01, 1.1))
add_panel_label(axes_bottom[3], 'I', xy=(-0.09, 1.05))
add_panel_label(axes_row4[0], 'J', xy=(-0.01, 1.1))
add_panel_label(axes_row4[3], 'K', xy=(-0.05, 1.05))
plt.savefig('fig3.pdf', format='pdf', bbox_inches='tight')